# End-to-End Tiered Data Management Pipeline (Google Colab Runner)

Reproducing a modular prototype of **Tiered Data Management** for LLM pre-training data based on:
> **"Data Science and Technology Towards AGI Part I: Tiered Data Management"**  
> *(arXiv:2602.09003)* — [https://arxiv.org/abs/2602.09003](https://arxiv.org/abs/2602.09003)

### Pipeline Overview:
- **Phase 1 (L1: Clean):** Heuristic text cleaning, length/ratio filtering & exact SHA-256 deduplication.
- **Phase 2 (L2: Selected):** Weak demo labeling, TF-IDF + numeric feature selector, scoring & selection.
- **Phase 3 (L3: Refined):** Deterministic offline MockLLM synthesis (cleaned text, Q&A, textbook chapters).
- **Phase 3.5 (L4: Organized):** Knowledge unit validation & structured export with provenance metadata.
- **Phase 4:** Cross-tier evaluation, retention analysis & multi-format report generation.

---  
## Step 1: Environment Setup & Repository Clone
Clone the official repository or pull the latest commits if already present.

In [ ]:
# Cell 1: Clone repository (recommended) or pull latest updates
import os

repo_url = "https://github.com/mahendravelagapudi099-wq/Ultra-Data.git"
target_dir = "/content/Ultra-Data"

if not os.path.exists(target_dir):
    !git clone {repo_url} {target_dir}
else:
    print(f"{target_dir} already exists. Pulling latest updates...")
    !cd {target_dir} && git pull origin main

# Optional: Mount Google Drive if syncing via Drive
# from google.colab import drive
# drive.mount('/content/drive')

---  
## Step 2: Navigate to Project Directory
Set the current working directory to the repository root.

In [ ]:
# Cell 2: Change directory into the project root
import os

candidate_paths = [
    "/content/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Dataa",
]

for path in candidate_paths:
    if os.path.exists(path):
        os.chdir(path)
        print(f"Active project root: {path}")
        break
else:
    print(f"Current working directory: {os.getcwd()}")

!pwd
!ls -la

---  
## Step 3: Install Required Dependencies
Installs Typer, Rich, Scikit-learn, Hugging Face Datasets, Pandas, and PyYAML.

In [ ]:
# Cell 3: Install required packages if missing
!pip install -q -r requirements.txt

---  
## Step 4: Configure Python Search Path (PYTHONPATH)
Ensures Python locates the `src` package and utility modules.

In [ ]:
# Cell 4: Set PYTHONPATH to project root
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["PYTHONPATH"] = "."
print(f"Project root (CWD): {project_root}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

---  
## Step 5: (Optional) Ingest Real Web Sample from Hugging Face
Streams real web documents from `openbmb/Ultra-FineWeb` (150 records) directly to `data/l0_raw/l0_real_sample.jsonl`.  
*Note: If offline, rate-limited, or interrupted, the next step automatically falls back to local substitute data.*

In [ ]:
# Cell 4.5: Fetch real web sample from openbmb/Ultra-FineWeb
# Writes records incrementally to disk with immediate flush
!PYTHONPATH=. python scripts/load_real_data.py -n 150

In [ ]:
import re
from pathlib import Path

script_path = Path('scripts/load_real_data.py')
content = script_path.read_text()

# Fix the AttributeError: replace args.count with the likely intended args.n based on typical parser conventions or the script's own -n flag
fixed_content = content.replace('args.count', 'args.n')

script_path.write_text(fixed_content)
print("Applied fix to scripts/load_real_data.py: replaced args.count with args.n")

### Retry Step 5: Ingest Real Web Sample
Now that the script is patched, we can attempt to fetch the 150 records from Hugging Face again.

In [ ]:
!PYTHONPATH=. python scripts/load_real_data.py -n 150

In [ ]:
import re
from pathlib import Path

script_path = Path('scripts/load_real_data.py')
content = script_path.read_text()

# 1. Fix the AttributeError
content = content.replace('args.count', 'args.n')

# 2. Broadly replace any mention of the 'train' split with 'en' to catch defaults
# We target patterns like: default='train', split='train', "train"
content = re.sub(r"(['\"])train(['\"])", r"\1en\2", content)

script_path.write_text(content)
print("Comprehensively updated all 'train' references to 'en' in scripts/load_real_data.py")

### Rerunning Ingestion and Full Pipeline
Now that we've updated the split name, let's ingest the real data and update the reports.

In [ ]:
!PYTHONPATH=. python scripts/load_real_data.py -n 150
!PYTHONPATH=. python scripts/run_phase1.py
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml
!PYTHONPATH=. python scripts/run_l3.py --config configs/l3_tiny.yaml
!PYTHONPATH=. python scripts/run_l4_export.py --config configs/l4_tiny.yaml
!PYTHONPATH=. python scripts/run_evaluation.py

In [ ]:
from pathlib import Path
import os

# Ensure we are in the correct directory
os.chdir('/content/Ultra-Data')

report = Path("reports/pipeline_summary.md")
if report.exists():
    print("### SUCCESS: Final Pipeline Report Generated ###\n")
    print(report.read_text())
else:
    print("Final report not found. Running Phase 4 to generate it...")
    !PYTHONPATH=. python scripts/run_evaluation.py
    if report.exists():
        print(report.read_text())
    else:
        print("Error: Could not generate report. Please check if Phase 1-3.5 completed successfully.")

---  
## Step 6: Phase 1 — Smart Ingestion & L1 Heuristic Filtering
Uses `scripts/run_phase1.py` as the smart orchestrator:  
- If `l0_real_sample.jsonl` exists from Step 5, ingests and filters the real web data.  
- If absent, automatically generates a synthetic local substitute dataset and filters it.  
- Applies NFKC normalization, boilerplate stripping, heuristic threshold gates, and SHA-256 deduplication.

In [ ]:
# Cell 5: Run Phase 1 via smart orchestrator (run_phase1.py)
!PYTHONPATH=. python scripts/run_phase1.py

---  
## Step 7: Phase 2 — L2 Model-Driven Selection
Trains a lightweight selector using TF-IDF n-grams and scaled text statistics (fitted strictly on the training split to avoid data leakage) to score and prioritize high-value educational tokens.

In [ ]:
# Cell 6: Run Phase 2 (L2 Model-Driven Selection)
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml

---  
## Step 8: Phase 3 & 3.5 — L3 Refinement & L4 Knowledge Export
- **Phase 3:** Uses deterministic offline `MockLLMProvider` to transform raw selected text into structured educational assets (overview article, conceptual Q&A pair, textbook chapter).  
- **Phase 3.5:** Applies structural quality gates and exports standardized knowledge units with UTC provenance metadata.

In [ ]:
# Cell 7: Run Phase 3 (Refinement) & Phase 3.5 (Organized Knowledge Export)
!PYTHONPATH=. python scripts/run_l3.py --config configs/l3_tiny.yaml
!PYTHONPATH=. python scripts/run_l4_export.py --config configs/l4_tiny.yaml

---  
## Step 9: Phase 4 — Cross-Tier Evaluation & Reporting
Computes tier-by-tier metrics (document counts, word counts, alphabetic/symbol ratios, duplicate counts, and transition rates) and saves Markdown, CSV, and JSON reports to `reports/`.

In [ ]:
# Cell 8: Run Phase 4 Evaluation across all tiers
!PYTHONPATH=. python scripts/run_evaluation.py

---  
## Step 10: Inspect Evaluation Report
Displays the final generated Markdown report summarizing quality progression across tiers.

In [ ]:
# Cell 9: Display the generated cross-tier summary report
from pathlib import Path

report_path = Path("reports/pipeline_summary.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Report not found at reports/pipeline_summary.md. Ensure Phase 4 ran successfully.")

---  
## Next Steps & References

- **Architecture & Methodology:** See [PHASES.md](PHASES.md) for a detailed mapping of pipeline components to the research paper.
- **Schema Specifications:** See [RESULTS_TEMPLATE.md](RESULTS_TEMPLATE.md) for exact column definitions across all tiers.
- **Paper Reference:** *"Data Science and Technology Towards AGI Part I: Tiered Data Management"* ([arXiv:2602.09003](https://arxiv.org/abs/2602.09003)).